In feature engineering, we havve to perfomm various transformations on the data to make it suitable for machine learning models. Some common feature engineering techniques include:
1. **Scaling**: This involves normalizing or standardizing the features to ensure that they are on the same scale. This is important for algorithms that are sensitive to the scale of the data, such as support vector machines and k-nearest neighbors.
2. **Encoding**: This is the process of converting categorical variables into numerical format. Common encoding techniques include one-hot encoding, label encoding, and ordinal encoding.
3. **Feature Creation**: This involves creating new features from existing ones. For example,you can create interaction features by multiplying two features together, or you can create polynomial features by raising a feature to a power.
4. **Feature Selection**: This is the process of selecting the most relevant features for the model. This can be done using techniques such as correlation analysis, mutual information, or using feature importance from models like random forests.
5. **Handling Missing Values**: This involves dealing with missing data in the dataset. Common techniques include imputation (filling in missing values with mean, median, or mode) or using algorithms that can handle missing values directly.
6. **Dimensionality Reduction**: This technique is used to reduce the number of features while retaining as much information as possible. Common methods include Principal Component Analysis (PCA) and t-Distributed Stochastic Neighbor Embedding (t-SNE).

By applying these feature engineering techniques, we can improve the performance of our machine learning models and make them more accurate and efficient.

but the normally we have to ***aplly all the techniques on columns individually*** , some colimns may require **scalling** while some may require **encoding**and the all **return arrays** which we have to concatenate together to form the final feature set for our model. This can be a **time-consuming and error-prone process**, especially when dealing with large datasets with many features.

# ColumnTransformer
To address this issue, we can use the `ColumnTransformer` from the `sklearn.compose` module. The `ColumnTransformer` allows us to apply different transformations to different columns of the dataset in a single step. This makes the feature engineering process more efficient and less error-prone.

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer # for handling missing values
from sklearn.preprocessing import OneHotEncoder # for encoding  nominal categorical variables
from sklearn.preprocessing import OrdinalEncoder # for encoding ordinal categorical variables

In [3]:
df = pd.read_csv('covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [7]:
X_train

,age,gender,fever,cough,city
17,40,Female,98.0,Strong,Delhi
16,69,Female,103.0,Mild,Kolkata
71,75,Female,104.0,Strong,Delhi
4,65,Female,101.0,Mild,Mumbai
35,82,Female,102.0,Strong,Bangalore
...,...,...,...,...,...
33,26,Female,98.0,Mild,Kolkata
30,15,Male,101.0,Mild,Delhi
73,34,Male,98.0,Strong,Kolkata
95,12,Female,104.0,Mild,Bangalore


## 1. Aam Zindagi

In [8]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [9]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [10]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [11]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [12]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

## Mentos Zindagi

In [13]:
from sklearn.compose import ColumnTransformer

In [ ]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')# passthrough means ohter columns will be left as it is

In [16]:
transformer.fit_transform(X_train).shape

(80, 7)

In [17]:
transformer.transform(X_test).shape

(20, 7)